# 02 · Corpus del Congreso: ParlaIbero-ES

Segundo notebook del preprocesado (E2). Aquí se descargan y se preparan las intervenciones del Pleno del Congreso de los Diputados que usa el proyecto para:
- entrenar el Word2Vec propio y analizar las palabras sin vector,
- sacar las sesiones del dataset propio (test anotado a mano y, quizá, un train),
- el análisis final por orador.

**Fuente: ParlaIbero-ES v2.0** (Universidad de Salamanca, Harvard Dataverse, [DOI 10.7910/DVN/OKHGAB](https://doi.org/10.7910/DVN/OKHGAB), licencia CC BY 4.0). Contiene todos los plenos del Congreso desde 1977-07-13 hasta 2025-12-11, ya divididos en intervenciones y con el orador identificado. Así no hay que extraer el texto de los PDF del Diario de Sesiones, que vienen a dos columnas y son difíciles de dividir por orador.

Cita: Rodrigues-Silveira, Rodrigo, 2026, "ParlaIbero-ES: Parliamentary Speeches from Spain (1977-2025)", https://doi.org/10.7910/DVN/OKHGAB, Harvard Dataverse.

## 0. Preparación

In [3]:
import hashlib
from pathlib import Path

import pandas as pd
import requests
from tqdm import tqdm

# El notebook se ejecuta desde la carpeta E2/, por eso las rutas empiezan por "..".
CARPETA_CONGRESO = Path("../data/raw/congreso")

## 1. Descarga

Se descargan de Harvard Dataverse:
- `ES_interventions.csv` (~1 GB): **el corpus**, una fila por intervención.
- La documentación en español: `README_es.md` (presentación y cifras del corpus), `data_dictionary_es.md` (qué significa cada columna) y `known_limitations_es.md` (limitaciones conocidas).

El corpus tiene más ficheros (padrón de diputados, informe de cómo se construyó, versiones en otros idiomas) que no hacen falta: el CSV de intervenciones ya trae el nombre y el partido de cada orador.

Los ficheros se guardan en `data/raw/congreso/`, que no se sube a GitHub. Cada fichero se pide por su identificador en Dataverse, que corresponde a la **versión 2.0**: así todo el grupo descarga exactamente los mismos datos. Los identificadores de las intervenciones de ParlaIbero solo son estables dentro de una versión, por eso hay que fijarla.

Como el CSV es grande, se descarga por trozos y al final se comprueba su **MD5** (una "huella" del fichero que publica Dataverse): si no coincide, el fichero se ha descargado mal.

In [4]:
# Nombre del fichero → (identificador en Dataverse, MD5 publicado). Versión 2.0 del corpus.
FICHEROS = {
    "ES_interventions.csv": (14215089, "264b24f08fb93cfc36a7967560d83010"),
    "README_es.md": (14215030, "f6400637b341ef4dfc14787413c20583"),
    "data_dictionary_es.md": (14205989, "462175f16f1b62adb80c55b5f6a65ca2"),
    "known_limitations_es.md": (14205663, "cb3e2561ddddaaa22fad47edb446e35d"),
}

URL_DATAVERSE = "https://dataverse.harvard.edu/api/access/datafile/{}"


def md5_de_fichero(ruta):
    """Calcula el MD5 leyendo el fichero por trozos (no hace falta cargarlo entero en memoria)."""
    md5 = hashlib.md5()
    with open(ruta, "rb") as fichero:
        for trozo in iter(lambda: fichero.read(1024 * 1024), b""):
            md5.update(trozo)
    return md5.hexdigest()


def descargar(nombre, identificador, md5_esperado):
    """Descarga un fichero de Dataverse por trozos, salvo que ya exista, y comprueba su MD5."""
    destino = CARPETA_CONGRESO / nombre
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(URL_DATAVERSE.format(identificador), stream=True, timeout=60) as respuesta:
            respuesta.raise_for_status()  # se para con un error si la descarga falla
            tamano = int(respuesta.headers.get("Content-Length", 0))
            with open(destino, "wb") as fichero, tqdm(total=tamano, unit="B", unit_scale=True, desc=nombre) as barra:
                for trozo in respuesta.iter_content(chunk_size=1024 * 1024):
                    fichero.write(trozo)
                    barra.update(len(trozo))

    if md5_de_fichero(destino) != md5_esperado:
        raise ValueError(f"{nombre}: el MD5 no coincide. Borra el fichero y vuelve a descargarlo.")
    print(f"OK: {nombre} ({destino.stat().st_size:,} bytes)")


for nombre, (identificador, md5_esperado) in FICHEROS.items():
    descargar(nombre, identificador, md5_esperado)

OK: ES_interventions.csv (1,058,266,687 bytes)
OK: README_es.md (10,259 bytes)
OK: data_dictionary_es.md (4,498 bytes)
OK: known_limitations_es.md (6,027 bytes)


## 2. Exploración

La documentación del corpus (`data_dictionary_es.md` y `known_limitations_es.md`) explica las 16 columnas. Lo más importante para nosotros:
- `dm_speech = 1` marca el habla parlamentaria, pero solo significa "no se ha demostrado que no sea discurso": pueden colarse documentos leídos en voz alta, sobre todo por la Presidencia.
- Los ministros sin escaño no tienen nombre normalizado ni partido.
- El texto sale de los PDF del Diario de Sesiones; en las sesiones antiguas pasó por OCR y puede tener errores.

El CSV pesa 1 GB, así que no se carga entero: primero se leen solo las columnas ligeras (todas menos el texto) para ver qué sesiones hay.

In [5]:
RUTA_CSV = CARPETA_CONGRESO / "ES_interventions.csv"
COLUMNAS_SIN_TEXTO = [
    "id_session", "id_int", "legislature", "session_number", "date", "session_type",
    "intervention_order", "speaker_raw", "id_dep", "speaker_name", "party", "dm_speech",
]

# dtype=str: todo se lee como texto (IDs, números de sesión...); así no se pierden ceros a la izquierda
meta = pd.read_csv(RUTA_CSV, usecols=COLUMNAS_SIN_TEXTO, dtype=str)
print(meta.shape)
meta.head(3)

(562514, 12)


,id_session,id_int,legislature,session_number,date,session_type,intervention_order,speaker_raw,id_dep,speaker_name,party,dm_speech
0,ES0010001,ES001000100000,Constituyente,1,1977-07-13,otros,0,NaN,NaN,NaN,NaN,0
1,ES0010001,ES001000100001,Constituyente,1,1977-07-13,otros,1,El señor RUBIO LLORENTE (Letrado de las Cortes),NaN,NaN,NaN,1
2,ES0010001,ES001000100002,Constituyente,1,1977-07-13,otros,2,El señor PRESIDENTE PROVISIONAL,NaN,NaN,NaN,1


In [6]:
# Sesiones por legislatura
sesiones = meta.groupby("id_session").agg(
    legislatura=("legislature", "first"), fecha=("date", "first"), intervenciones=("id_int", "size")
)
sesiones.groupby("legislatura").agg(
    sesiones=("fecha", "size"), desde=("fecha", "min"), hasta=("fecha", "max")
).sort_values("desde")

,sesiones,desde,hasta
legislatura,,,
Constituyente,197,1977-07-13,1979-03-07
I,259,1979-03-23,1982-11-10
II,282,1982-11-18,1986-06-24
III,208,1986-07-15,1989-11-20
IV,265,1989-11-21,1993-06-23
V,197,1993-06-29,1996-03-25
VI,286,1996-03-27,2000-04-04
VII,310,2000-04-05,2004-04-01
VIII,315,2004-04-02,2008-03-31


In [7]:
print(meta["dm_speech"].value_counts(), "\n")
print(meta["session_type"].value_counts())

dm_speech
1    558940
0      3574
Name: count, dtype: int64 

session_type
ordinaria    534320
otros         28194
Name: count, dtype: int64


### 2.1 La XV legislatura

Para elegir las sesiones nos centramos en la **XV legislatura** (agosto de 2023 – diciembre de 2025):
- Son PDF digitales recientes, sin los errores de OCR de las sesiones antiguas.
- Hablan los mismos políticos que en los debates de 2023 de FallacyES-Political (Sánchez, Feijóo, Abascal, Díaz), y empieza justo después de ellos, sin solaparse. Así el análisis por político se puede comparar.

Se lee el CSV por trozos de 50.000 filas y se guardan solo las de la XV, para no tener el GB entero en memoria.

In [8]:
def leer_legislatura(legislatura):
    """Lee el CSV por trozos y se queda solo con las filas de esa legislatura."""
    trozos = []
    for trozo in pd.read_csv(RUTA_CSV, dtype=str, chunksize=50_000):
        trozos.append(trozo[trozo["legislature"] == legislatura])
    return pd.concat(trozos, ignore_index=True)


xv = leer_legislatura("XV")
habla = xv[xv["dm_speech"] == "1"]  # sin el material previo (carátula, sumario)
print("Filas de la XV:", len(xv), "| de habla:", len(habla), "| sesiones:", xv["id_session"].nunique())

Filas de la XV: 26105 | de habla: 25946 | sesiones: 159


In [9]:
palabras = habla["text"].str.split().str.len()
print("Palabras en total:", f"{palabras.sum():,}")
print("Palabras por sesión (media):", f"{palabras.groupby(habla['id_session']).sum().mean():,.0f}")

fechas = pd.to_datetime(xv.groupby("id_session")["date"].first())
print("Sesiones por año:", fechas.dt.year.value_counts().sort_index().to_dict())
print("Día de la semana:", fechas.dt.day_name().value_counts().to_dict())

# Palabras por intervención
palabras.describe().round(0)

Palabras en total: 8,358,647
Palabras por sesión (media): 52,570
Sesiones por año: {2023: 15, 2024: 76, 2025: 68}
Día de la semana: {'Tuesday': 63, 'Wednesday': 61, 'Thursday': 34, 'Friday': 1}


count    25946.0
mean       322.0
std        647.0
min          1.0
25%         17.0
50%         44.0
75%        297.0
max      13942.0
Name: text, dtype: float64

Así es una sesión por dentro. La fila con `intervention_order = 0` es el material previo (carátula y sumario, con el orden del día); después, cada fila es una intervención. La Presidencia interviene muchísimo, casi siempre con frases cortas para dar la palabra.

In [10]:
una_sesion = xv[xv["id_session"] == "ES0160101"]
material_previo = una_sesion.loc[una_sesion["intervention_order"] == "0", "text"].iloc[0]
print(material_previo[:700], "...\n")
una_sesion[["intervention_order", "speaker_raw", "speaker_name", "party", "text"]].head(8)

CORTES GENERALES DIARIO DE SESIONES DEL CONGRESO DE LOS DIPUTADOS Año 2025 XV LEGISLATURA Núm. 101 Pág. 1 PRESIDENCIA DE LA EXCMA. SRA. D.ª FRANCINA ARMENGOL SOCIAS Sesión plenaria núm. 97 celebrada el miércoles 12 de marzo de 2025 ORDEN DEL DÍA:

Preguntas .................................................................................................................................. 6 Interpelaciones urgentes:

— Del Grupo Parlamentario Mixto (señor Rego Candamil), relativa a la posición del Gobierno español ante los anuncios intervencionistas, belicistas, anexionistas y de apoyo a la limpieza étnica de los Estados Unidos de América. (Número de expediente 172/000094) ..................... ...



,intervention_order,speaker_raw,speaker_name,party,text
14947,0,NaN,NaN,NaN,CORTES GENERALES DIARIO DE SESIONES DEL CONGRE...
14948,1,La señora PRESIDENTA,"ARMENGOL SOCIAS, FRANCINA",PSIB-PSOE,Reanudamos la sesión.\n\nBuenos días a todos y...
14949,2,La señora GAMARRA RUIZ-CLAVIJO,"GAMARRA RUIZ-CLAVIJO, CONCEPCIÓN",PP,"Gracias, presidenta.\n\nSeñora Montero, ahora ..."
14950,3,La señora PRESIDENTA,"ARMENGOL SOCIAS, FRANCINA",PSIB-PSOE,"Muchas gracias, señora Gamarra.\n\nSeñora vice..."
14951,4,La señora VICEPRESIDENTA PRIMERA Y MINISTRA HA...,"MONTERO CUADRADO, MARÍA JESÚS",PSOE,"Muchas gracias, señora presidenta.\n\nSeñora G..."
14952,5,La señora PRESIDENTA,"ARMENGOL SOCIAS, FRANCINA",PSIB-PSOE,"Muchas gracias, señora vicepresidenta.\n\nSeño..."
14953,6,La señora GAMARA RUIZ-CLAVIJO,"GAMARRA RUIZ-CLAVIJO, CONCEPCIÓN",PP,"Señora Montero, si todo esto es tan bueno, tan..."
14954,7,La señora PRESIDENTA,"ARMENGOL SOCIAS, FRANCINA",PSIB-PSOE,Muchas gracias.\n\nSeñora vicepresidenta primera.


In [11]:
# Las sesiones de control al Gobierno (preguntas de la oposición, los miércoles) llevan "Preguntas"
# en el orden del día del material previo. Son las más confrontacionales.
material_previo_por_sesion = xv[xv["intervention_order"] == "0"].set_index("id_session")["text"]
es_de_control = material_previo_por_sesion.str.contains("Preguntas")
print("Sesiones de control:", es_de_control.sum(), "de", len(material_previo_por_sesion))

Sesiones de control: 52 de 159


### 2.2 Acotaciones

El Diario de Sesiones anota entre paréntesis lo que pasa en la sala: aplausos, rumores, risas, interrupciones de otros diputados... No son palabras del orador y habrá que quitarlas. Pero no todo lo que va entre paréntesis es una acotación: también hay siglas dentro del discurso, como *(EAJ-PNV)*.

In [12]:
import re
from collections import Counter

entre_parentesis = Counter(
    fragmento for texto in habla["text"] for fragmento in re.findall(r"\([^()]{1,80}\)", texto)
)
entre_parentesis.most_common(30)

[('(Aplausos)', 19125),
 ('(Rumores)', 2507),
 ('(Pausa)', 1932),
 ('(aplausos)', 1866),
 ('(rumores)', 537),
 ('(Risas)', 534),
 ('(Protestas)', 516),
 ('(EAJ-PNV)', 451),
 ('(VOTACIÓN)', 319),
 ('(risas)', 195),
 ('(El señor vicepresidente, Rodríguez Gómez de Celis, ocupa la Presidencia)',
  179),
 ('(protestas)', 178),
 ('(La señora presidenta ocupa la Presidencia)', 166),
 ('(Continúan los rumores)', 139),
 ('(Risas y aplausos)', 119),
 ('(Rumores.―Aplausos)', 114),
 ('(Aplausos.―Rumores)', 111),
 ('(Aplausos.—Rumores)', 103),
 ('(muestra un documento)', 85),
 ('(Rumores.—Aplausos)', 82),
 ('(Muestra un documento)', 81),
 ('(Continúan los aplausos)', 74),
 ('(Asentimiento)', 72),
 ('(Aplausos.―Varios señores diputados: ¡Muy bien!)', 67),
 ('(Continúan las protestas)', 64),
 ('(Protestas.—Aplausos)', 64),
 ('(Aplausos.—Varios señores diputados: ¡Muy bien!)', 59),
 ('(Risas.—Aplausos)', 52),
 ('(Protestas.―Aplausos)', 51),
 ('(Denegación)', 50)]

In [13]:
# Un ejemplo: la acotación incluye lo que grita otro diputado
buscado = "Tellado Filgueira: ¡Oooh!"
texto = habla[habla["text"].str.contains(buscado, regex=False)]["text"].iloc[0]
posicion = texto.find(buscado)
print(texto[posicion - 250 : posicion + 150])

, dimitiendo de lo que debería ser su tarea parlamentaria, uno tiene la duda de si sirve de algo interpelarles, porque nada de lo que ustedes traen aquí tiene su origen aquí.

Así que, efectivamente, vamos a votar en contra de su petición. (El señor Tellado Filgueira: ¡Oooh!). 

Y pedimos al resto de grupos que la rechacen porque esto no hace más que inaugurar por todo lo alto la nueva temporada d


### 2.3 Oradores

`speaker_raw` es el orador tal como sale en el acta. Hay que distinguir la **Presidencia del Congreso** (quien modera: *La señora PRESIDENTA*, los vicepresidentes de la Mesa) del **Gobierno** (*PRESIDENTE DEL GOBIERNO*, *VICEPRESIDENTA PRIMERA Y MINISTRA…*): las dos llevan la palabra "PRESIDENT".

In [14]:
con_president = habla[habla["speaker_raw"].str.contains("PRESIDENT", na=False)]
con_president.groupby("speaker_raw").agg(
    intervenciones=("id_int", "size"), nombre=("speaker_name", "first"), partido=("party", "first")
).sort_values("intervenciones", ascending=False).head(12)

,intervenciones,nombre,partido
speaker_raw,,,
La señora PRESIDENTA,10479,"ARMENGOL SOCIAS, FRANCINA",PSIB-PSOE
El señor VICEPRESIDENTE (Rodríguez Gómez de Celis),4416,"RODRÍGUEZ GÓMEZ DE CELIS, ALFONSO",PSOE
La señora VICEPRESIDENTA PRIMERA Y MINISTRA DE HACIENDA (Montero Cuadrado),377,"MONTERO CUADRADO, MARÍA JESÚS",PSOE
El señor PRESIDENTE DEL GOBIERNO (Sánchez Pérez-Castejón),280,"SÁNCHEZ PÉREZ-CASTEJÓN, PEDRO",PSOE
La señora VICEPRESIDENTA SEGUNDA Y MINISTRA DE TRABAJO Y ECONOMÍA SOCIAL (Díaz Pérez),90,"DÍAZ PÉREZ, YOLANDA",SUMAR
La señora VICEPRESIDENTA TERCERA Y MINISTRA PARA LA TRANSICIÓN ECOLÓGICA Y EL RETO DEMOGRÁFICO (Aagesen Muñoz),69,NaN,NaN
El señor PRESIDENTE,43,"RODRÍGUEZ GÓMEZ DE CELIS, ALFONSO",PSOE
La señora VICEPRESIDENTA PRIMERA DEL GOBIERNO Y MINISTRA DE HACIENDA (Montero Cuadrado),41,"MONTERO CUADRADO, MARÍA JESÚS",PSOE
La señora VICEPRESIDENTA TERCERA Y MINISTRA PARA LA TRANSICIÓN ECOLÓGICA Y EL RETO DEMOGRÁFICO (Ribera Rodríguez),34,"RIBERA RODRÍGUEZ, TERESA",PSOE


In [15]:
ministros = habla[habla["speaker_raw"].str.contains("MINISTR", na=False)]
print("Intervenciones de ministros:", len(ministros), "| sin nombre ni partido (sin escaño):", ministros["speaker_name"].isna().sum())
print("Intervenciones sin nombre normalizado en total:", habla["speaker_name"].isna().sum())

habla["party"].value_counts()

Intervenciones de ministros: 2139 | sin nombre ni partido (sin escaño): 463
Intervenciones sin nombre normalizado en total: 564


party
PSIB-PSOE        10504
PSOE              6944
PP                2261
SUMAR             1410
VOX               1269
ERC                612
EAJ-PNV            533
JxCAT-JUNTS        510
EH Bildu           425
BNG                323
CCa                169
UPN                159
PSC-PSOE           122
PSE-EE (PSOE)       90
PsdeG-PSOE          40
PSN-PSOE            11
Name: count, dtype: int64

In [16]:
# Caracteres que la extracción del PDF no pudo descifrar (según la documentación, 606 filas en todo el corpus)
print("Filas de la XV con caracteres sin descifrar:", habla["text"].str.contains("[-]").sum())

Filas de la XV con caracteres sin descifrar: 0


### 2.4 Resumen de la exploración

- El corpus tiene 3.559 sesiones (1977–2025). La **XV legislatura** tiene 159 sesiones y unos 8,4 millones de palabras, unas 52.000 por sesión: con 40 sesiones habría unos 2 millones para el Word2Vec.
- 52 de las 159 sesiones son **sesiones de control** (preguntas al Gobierno): las más confrontacionales, buenas candidatas para el test propio.
- **Acotaciones**: hay muchas (*(Aplausos)* aparece unas 19.000 veces) y a veces incluyen lo que grita otro diputado. Hay que quitarlas sin quitar los paréntesis legítimos como *(EAJ-PNV)*.
- **Presidencia**: la presidenta del Congreso (Armengol) tiene unas 10.500 intervenciones y el corpus le asigna partido (PSIB-PSOE). Si no se separa, el análisis por partido sale muy desviado. Se detecta con `speaker_raw`, distinguiéndola del Gobierno (*PRESIDENTE DEL GOBIERNO*, *VICEPRESIDENTA… Y MINISTRA*).
- **Ministros sin escaño**: no tienen nombre normalizado ni partido.
- El PSOE aparece con varias etiquetas regionales (PSOE, PSC-PSOE, PSE-EE (PSOE)…): para el análisis por partido habrá que juntarlas.
- La XV no tiene caracteres sin descifrar.

## 3. Elección de sesiones

Cada sesión elegida tiene un **uso**, y todas sus intervenciones lo heredan (nunca se mezclan intervenciones de una misma sesión entre usos):

| Uso | Sesiones | Para qué |
|---|---|---|
| `test_propio` | las **3 sesiones de control más recientes** | Se anotarán a mano: es el test del detector en el Congreso. Son las más confrontacionales (preguntas de la oposición al Gobierno), así que tendrán más falacias |
| `general` | **35 sesiones al azar** de las demás de la XV | Entrenar el Word2Vec propio, analizar las palabras sin vector y el análisis final por político |

**El Word2Vec (y cualquier cosa que aprenda del texto) se entrena solo con las sesiones `general`**, para que no haya visto el texto del test.

Se descartó un tercer uso, `train_propio` (sesiones del Congreso anotadas para entrenar): FallacyES-Political ya aporta unas 1.300 falacias de debates políticos españoles para entrenar, y anotar otro train a mano no compensa el esfuerzo.

Se toman solo 3 sesiones de test porque anotar es caro: una sesión de control tiene de media unas 110 intervenciones (sin contar la Presidencia).

In [17]:
SEMILLA = 42  # semilla fija: la elección sale igual cada vez que se ejecuta
NUM_SESIONES_TEST_PROPIO = 3
NUM_SESIONES_GENERAL = 35

fecha_de_cada_sesion = xv.groupby("id_session")["date"].first().sort_values()
sesiones_de_control = es_de_control[es_de_control].index  # de la sección 2.1

# test_propio: las sesiones de control más recientes
control_por_fecha = fecha_de_cada_sesion[fecha_de_cada_sesion.index.isin(sesiones_de_control)]
sesiones_test_propio = control_por_fecha.index[-NUM_SESIONES_TEST_PROPIO:]

# general: al azar entre todas las demás
resto = fecha_de_cada_sesion.index.difference(sesiones_test_propio)
sesiones_general = pd.Series(resto).sample(NUM_SESIONES_GENERAL, random_state=SEMILLA)

uso_de_cada_sesion = {sesion: "test_propio" for sesion in sesiones_test_propio}
uso_de_cada_sesion.update({sesion: "general" for sesion in sesiones_general})

print("Sesiones test_propio:", fecha_de_cada_sesion[sesiones_test_propio].to_dict())

Sesiones test_propio: {'ES0160153': '2025-11-19', 'ES0160155': '2025-11-26', 'ES0160158': '2025-12-10'}


In [18]:
# Cuántas sesiones hay de cada uso y cuántas son de control
sesiones_elegidas = pd.DataFrame({"uso": pd.Series(uso_de_cada_sesion)})
sesiones_elegidas["de_control"] = sesiones_elegidas.index.isin(sesiones_de_control)
sesiones_elegidas.groupby("uso")["de_control"].agg(sesiones="size", de_control="sum")

,sesiones,de_control
uso,,
general,35,9
test_propio,3,3


In [19]:
# Intervenciones y palabras de cada uso, sin contar la Presidencia del Congreso
# (aquí se separa de forma provisional; la marca definitiva es_presidencia se hace al limpiar)
elegidas = habla[habla["id_session"].isin(uso_de_cada_sesion)].copy()
elegidas["uso"] = elegidas["id_session"].map(uso_de_cada_sesion)
orador = elegidas["speaker_raw"].fillna("")
habla_la_presidencia = orador.str.contains("PRESIDENT") & ~orador.str.contains("GOBIERNO|MINISTR")

elegidas[~habla_la_presidencia].groupby("uso").agg(
    intervenciones=("id_int", "size"),
    palabras=("text", lambda textos: textos.str.split().str.len().sum()),
)

,intervenciones,palabras
uso,,
general,2149,1484401
test_propio,323,172971


In [20]:
# Las sesiones general se reparten por toda la legislatura
fechas_general = pd.to_datetime(fecha_de_cada_sesion[sesiones_general])
print("Sesiones general por año:", fechas_general.dt.year.value_counts().sort_index().to_dict())
print("Comprobación: ninguna sesión con dos usos:", len(set(sesiones_general) & set(sesiones_test_propio)) == 0)

Sesiones general por año: {2023: 2, 2024: 17, 2025: 16}
Comprobación: ninguna sesión con dos usos: True


## 4. Filtrado: sesiones elegidas, solo habla y sin la Mesa

Se quedan solo:
1. Las intervenciones de las 38 sesiones elegidas, con su `uso`.
2. Las filas de habla (`dm_speech == 1`): fuera la carátula y el sumario.
3. **Sin las intervenciones de la Mesa del Congreso** (Presidencia, vicepresidencias, Mesa de Edad y secretarías).

**Por qué se quita la Mesa:** modera la sesión ("Muchas gracias, señora Gamarra. Señora vicepresidenta primera."), no argumenta, así que no puede contener falacias. No sirve como ejemplo para los modelos (sería un "sin falacia" tan distinto de un discurso que el modelo aprendería un atajo), solo aporta fórmulas repetidas al Word2Vec y en el análisis por partido la presidenta saldría como la diputada del PSOE con más intervenciones. El orden de las intervenciones se conserva en el ID, así que la sesión se puede reconstruir sin ellas.

**Cuidado con la regla:** el Gobierno también lleva la palabra "PRESIDENT" (*PRESIDENTE DEL GOBIERNO*, *VICEPRESIDENTA PRIMERA Y MINISTRA…*) y **no** es la Mesa: se queda.

In [21]:
def es_de_la_mesa(orador):
    """True si habla la Mesa del Congreso (Presidencia, vicepresidencias, Mesa de Edad, secretarías).
    El Gobierno también lleva "PRESIDENT" (PRESIDENTE DEL GOBIERNO, VICEPRESIDENTA... Y MINISTRA)
    y no es la Mesa."""
    if "GOBIERNO" in orador or "MINISTR" in orador:
        return False
    return "PRESIDENT" in orador or "SECRETARI" in orador


congreso = xv[xv["id_session"].isin(uso_de_cada_sesion) & (xv["dm_speech"] == "1")].copy()
congreso["uso"] = congreso["id_session"].map(uso_de_cada_sesion)

habla_la_mesa = congreso["speaker_raw"].fillna("").map(es_de_la_mesa)
print("Intervenciones de habla en las sesiones elegidas:", len(congreso))
print("De la Mesa (se quitan):", habla_la_mesa.sum())

# Qué se quita
congreso.loc[habla_la_mesa, "speaker_raw"].value_counts()

Intervenciones de habla en las sesiones elegidas: 6024
De la Mesa (se quitan): 3556


speaker_raw
La señora PRESIDENTA                                      2463
El señor VICEPRESIDENTE (Rodríguez Gómez de Celis)        1075
El señor VICEPRESIDENTE                                      5
El señor SECRETARIO (Pisarello Prados)                       4
El señor VICEPRESIDENTE (Bermúdez de Castro Fernández)       3
El señor PRESIDENTE                                          2
El señor VICEPRESIDENTE (Rodríguez Gómez de Celís)           2
La señora VICEPRESIDENTA (Gil de Reboleño Lastortres)        2
Name: count, dtype: int64

In [22]:
# Comprobación: el Gobierno con "PRESIDENT" en el cargo se queda
gobierno = congreso.loc[~habla_la_mesa, "speaker_raw"].fillna("")
gobierno[gobierno.str.contains("PRESIDENT")].value_counts().head(6)

speaker_raw
La señora VICEPRESIDENTA PRIMERA Y MINISTRA DE HACIENDA (Montero Cuadrado)                                        91
El señor PRESIDENTE DEL GOBIERNO (Sánchez Pérez-Castejón)                                                         67
La señora VICEPRESIDENTA SEGUNDA Y MINISTRA DE TRABAJO Y ECONOMÍA SOCIAL (Díaz Pérez)                             23
La señora VICEPRESIDENTA TERCERA Y MINISTRA PARA LA TRANSICIÓN ECOLÓGICA Y EL RETO DEMOGRÁFICO (Aagesen Muñoz)    20
La señora VICEPRESIDENTA PRIMERA DEL GOBIERNO Y MINISTRA DE HACIENDA (Montero Cuadrado)                           11
La señora VICEPRESIDENTA PRIMERA Y MINISTRA DE ECONOMÍA, COMERCIO Y EMPRESA (Calviño Santamaría)                   4
Name: count, dtype: int64

In [23]:
congreso = congreso[~habla_la_mesa]

palabras = congreso["text"].str.split().str.len()
congreso.assign(palabras=palabras).groupby("uso").agg(
    sesiones=("id_session", "nunique"), intervenciones=("id_int", "size"), palabras=("palabras", "sum")
)

,sesiones,intervenciones,palabras
uso,,,
general,35,2146,1483531
test_propio,3,322,172628


## 5. Limpieza del texto

El Diario de Sesiones mezcla con las palabras del orador cosas que **no dijo él**. Se quitan; el texto sin tocar se guardará en `texto_original`.

| Qué se quita | Ejemplo | Cuánto hay |
|---|---|---|
| **Acotaciones**: todo lo que va entre paréntesis | *(Aplausos)*, *(Rumores)*, *(El señor Tellado Filgueira: ¡Oooh!)*, *(muestra un documento)* | unas 7.000 |
| **Títulos del orden del día** pegados al final de la intervención: líneas casi enteras en mayúsculas | *PROPOSICIONES NO DE LEY:* / *— DEL DIPUTADO DON SANTIAGO ABASCAL CONDE… QUE FORMULA AL SEÑOR PRESIDENTE DEL GOBIERNO: ¿…?* | 26 líneas |
| **Marcas de palabras retiradas**: el corchete y el número de nota (las palabras se quedan) | *Ahora te mandan [golpismo judicial]1.* → *Ahora te mandan golpismo judicial.* | 36 |

**Por qué se quita todo lo que va entre paréntesis:** casi todo es del editor del Diario (reacciones, gritos de otros diputados, gestos, *(sic)*). Solo una quincena son palabras del orador, como *(UE)* o *(DANA)*; perderlas no importa y la regla queda muy simple.

**El punto que cierra la acotación:** el Diario escribe *"¿le suena lo que le digo? (Aplausos). Señora Montesinos…"*. Si la acotación va tras un final de frase, se quita con su punto, para no dejar *"¿le suena lo que le digo? . Señora…"*. Si va en mitad de la frase (*"ni un solo ministro (aplausos), porque…"*), se quita solo el paréntesis.

**Por qué se conservan las palabras retiradas:** la Presidencia puede retirar del Diario expresiones ofensivas, que quedan entre corchetes con una nota. Se dijeron de verdad y suelen ser insultos (*"[sinvergüenzas]1"*, *"[un Gobierno corrupto] 8"*): justo lo que buscamos al detectar *ad hominem*.

**Qué se deja:**
- **Otras lenguas** (catalán, euskera, gallego): son el 1,7 % de las palabras, casi siempre unas pocas frases al empezar o terminar una intervención en castellano. Quitarlas exigiría detectar el idioma frase a frase. Si molestan, se pueden filtrar al dividir en oraciones (notebook 03).
- **Fórmulas de trámite** (*"Gracias, presidenta."*, *"Muchas gracias."*): al dividir en oraciones quedan como frases sueltas, iguales para todos los oradores, así que no sesgan nada.
- Los puntos suspensivos de las citas, *[…]*: son del orador.

Al final se deja un solo espacio entre palabras, igual que en el notebook 01.

In [ ]:
# Acotación tras un final de frase o una coma: "¿le suena? (Aplausos). Señora..."
# Se quita con el punto o la coma que la cierra, para no dejar "¿le suena? . Señora..."
ACOTACION_TRAS_PUNTO = r"(?:^|(?<=[.?!…,]))\s*\([^()]*\)[.,]?"
# Acotación en mitad de una frase: "ni un solo ministro (aplausos), porque..."
ACOTACION_EN_MEDIO = r"\s*\([^()]*\)"
# Palabras retiradas del Diario por la Presidencia: "[golpismo judicial]1" o "[indecente] 1"
PALABRAS_RETIRADAS = r"\[([^\]]+)\] ?\d+"


def es_titulo_del_orden_del_dia(linea):
    """True si la línea va casi entera en mayúsculas, como "PROPOSICIONES NO DE LEY:"
    o "— DEL DIPUTADO DON ... QUE FORMULA ...". No cuenta lo que va entre paréntesis,
    porque los títulos acaban en "(Número de expediente 121/000031)"."""
    letras = [letra for letra in re.sub(r"\([^()]*\)", "", linea) if letra.isalpha()]
    if len(letras) < 10:  # líneas muy cortas ("¡NO!", "UE") no son títulos
        return False
    return sum(letra.isupper() for letra in letras) / len(letras) > 0.8


def quitar_titulos_del_orden_del_dia(texto):
    lineas = texto.split("\n")
    return "\n".join(linea for linea in lineas if not es_titulo_del_orden_del_dia(linea))


def quitar_acotaciones(texto):
    texto = re.sub(ACOTACION_TRAS_PUNTO, "", texto)
    return re.sub(ACOTACION_EN_MEDIO, "", texto)


def limpiar_intervencion(texto):
    """Quita los títulos del orden del día, las acotaciones y las marcas de palabras retiradas,
    y deja un solo espacio entre palabras."""
    texto = quitar_titulos_del_orden_del_dia(texto)  # primero, porque va por líneas
    texto = quitar_acotaciones(texto)
    texto = re.sub(PALABRAS_RETIRADAS, r"\1", texto)
    return " ".join(texto.split())


# Ejemplos sacados del corpus
for prueba in ["¿le suena lo que le digo?\n\n(Aplausos). Señora Montesinos",
               "ni un solo ministro (aplausos), porque ustedes",
               "altos cargos (aplausos) y también",
               "(Comienza su intervención en euskera). Eskerrik asko.",
               "¡Qué progresismo! (Varios diputados: ¡Muy bien!— Aplausos). Señor Bolaños",
               "Ahora te mandan [golpismo judicial]1. Es usted un [indecente] 1.",
               "Muchas gracias.\nPROPOSICIONES NO DE LEY:\n— DEL GRUPO PARLAMENTARIO VOX, SOBRE LA ESTRATEGIA DEL GOBIERNO. (Número de expediente 173/000136)."]:
    print(repr(prueba), "\n   →", repr(limpiar_intervencion(prueba)))

In [ ]:
congreso["texto"] = congreso["text"].map(limpiar_intervencion)

palabras_antes = congreso["text"].str.split().str.len().sum()
palabras_despues = congreso["texto"].str.split().str.len().sum()
print(f"Palabras: {palabras_antes} → {palabras_despues} ({(palabras_antes - palabras_despues) / palabras_antes:.1%} menos)")
print("Intervenciones vacías tras limpiar:", (congreso["texto"] == "").sum())

In [ ]:
# Comprobación: qué paréntesis quedan
for texto in congreso.loc[congreso["texto"].str.contains(r"[()]"), "texto"]:
    for trozo in re.findall(r".{0,40}[()].{0,30}", texto):
        print("   ", trozo)

Los paréntesis que quedan son letras de apartados de leyes (*"el artículo 80.1.b) de la LOSU"*), que sí son palabras del orador, y una errata del Diario con un paréntesis de más: *"(Rumores.―El señor Tellado Filgueira: ¡Ábalos!).―Aplausos)."*. Se deja así: es un caso.

In [ ]:
# Un trozo de una intervención antes y después de limpiar
ejemplo = congreso.loc[congreso["text"].str.contains(r"golpismo judicial\]2"), ["text", "texto"]].iloc[0]
inicio = ejemplo["text"].find("Seguramente")
print(ejemplo["text"][inicio:inicio + 500])
print("\n" + "=" * 100 + "\n")
inicio = ejemplo["texto"].find("Seguramente")
print(ejemplo["texto"][inicio:inicio + 450])

### 5.1 Resumen

- Se quita el **1,4 % de las palabras** (acotaciones y títulos). Ninguna intervención queda vacía.
- Quedan en el texto, a propósito: las otras lenguas (1,7 % de las palabras), las fórmulas de trámite y las palabras retiradas por la Presidencia.
- Aún hay intervenciones de una o dos palabras (*"Sí, prometo."* al tomar posesión del escaño). Son pocas y no estorban.